# 🔺 Delta Lake — Z-Ordering

---

## 1. O que é Z-Ordering?
![Foto](./Fotos/15.png)
![Foto](./Fotos/16.png)
![Foto](./Fotos/17.png)
O Z-Ordering é uma técnica de otimização inspirada na **Curva Z** (Zorro) que decide de forma inteligente  
**quais linhas colocar em quais arquivos**, co-localizando dados relacionados no mesmo Parquet.

Diferente do Partitioning, que cria subdiretórios físicos por valor, o Z-Ordering:
- **Não cria subdiretórios** — todos os arquivos ficam na raiz da tabela
- **Reorganiza as linhas dentro dos arquivos** usando o algoritmo Z
- **Funciona com colunas de alta cardinalidade** onde o particionamento criaria small files

---

### O algoritmo Z — como ele conecta os dados

O algoritmo traça uma linha em formato de Z pelos valores de duas colunas,  
conectando pontos próximos no espaço multidimensional:

```
  Col B
    4  │ (1,4) (2,4) │ (3,4) (4,4) │
    3  │ (1,3) (2,3) │ (3,3) (4,3) │
       │─────────────────────────────
    2  │ (1,2) (2,2) │ (3,2) (4,2) │
    1  │ (1,1) (2,1) │ (3,1) (4,1) │
       └──────────────────────────────► Col A
            1    2        3    4

  A linha Z percorre: (1,1)→(2,1)→(1,2)→(2,2)→(3,1)→(4,1)→(3,2)→(4,2)→...
  Cada grupo de 4 pontos consecutivos nessa linha → 1 arquivo Parquet
```

> O resultado são arquivos que contêm pontos **próximos no espaço de duas colunas**,  
> não apenas próximos em uma única dimensão como no particionamento.

---

### Z-Ordering vs Partitioning — comparação de leitura

Para a query: `WHERE ColA = 4 OR ColB = 3`

```
  PARTITIONING por ColA e ColB (range 1-4 / 5-8)
  ─────────────────────────────────────────────────
  ColA = 4  → lê 2 arquivos (subfolder ColA=3-4)
  ColB = 3  → lê 7 arquivos (todos os subfolders com ColB=3)
  Total:      9 arquivos lidos
  Falsos positivos: 21 linhas lidas desnecessariamente
              (estão no mesmo arquivo que os dados relevantes)

  Z-ORDERING por ColA e ColB
  ─────────────────────────────────────────────────
  ColA = 4  → lê 4 arquivos (stats do _delta_log eliminam os demais)
  ColB = 3  → lê 3 arquivos adicionais
  Total:      7 arquivos lidos
  Falsos positivos: significativamente menores
```

```
  Resultado: Z-Ordering leu 7 arquivos vs 9 do Partitioning
  Em escala (bilhões de linhas), essa diferença é enorme.
```

> **Por que o Z-Ordering tem menos falsos positivos?**  
> Como linhas com valores próximos em AMBAS as colunas ficam no mesmo arquivo,  
> as estatísticas (`minValues`/`maxValues`) do `_delta_log` são muito mais precisas  
> para eliminar arquivos que certamente não contêm os dados buscados.

---

### Partitioning vs Z-Ordering — quando usar cada um

```
  ┌──────────────────────────────────────────────────────────────┐
  │                                                              │
  │  Coluna com BAIXA cardinalidade + tabela > 1TB               │
  │  (ex: Year, Country, Category)                               │
  │          │                                                   │
  │          ▼                                                   │
  │    PARTITIONING                                              │
  │    Subdiretórios físicos | Benefício claro em scans parciais │
  │                                                              │
  │  Coluna com ALTA cardinalidade                               │
  │  (ex: UserId, Origin, FlightNum, colunas usadas em JOINs)    │
  │          │                                                   │
  │          ▼                                                   │
  │    Z-ORDERING                                                │
  │    Sem subdiretórios | Co-localiza dados | Menos falsos pos. │
  │                                                              │
  │  Caso mais comum em produção: PARTITIONING + Z-ORDERING      │
  │  Partição por coluna de baixa cardinalidade (ex: Year)       │
  │  Z-Order por coluna de alta cardinalidade dentro da partição │
  └──────────────────────────────────────────────────────────────┘
```

---

## 2. Carregando os dados

In [ ]:
%%sql
DROP TABLE IF EXISTS flight_data_z;

CREATE TABLE flight_data_z (
    Year INT, Month INT, DayOfMonth INT, DayOfWeek INT, DepTime STRING, CRSDepTime INT, ArrTime STRING, CRSArrTime INT, UniqueCarrier STRING, FlightNum INT, TailNum STRING, ActualElapsedTime STRING, CRSElapsedTime STRING, AirTime STRING, ArrDelay STRING, DepDelay STRING, Origin STRING, Dest STRING, Distance INT, TaxiIn STRING, TaxiOut STRING, Cancelled INT, CancellationCode STRING, Diverted INT, CarrierDelay STRING, WeatherDelay STRING, NASDelay STRING, SecurityDelay STRING, LateAircraftDelay STRING
);

In [ ]:
%python
# Ingere todos os anos disponíveis (glob 2*.csv) em uma única tabela sem partição
spark.read.csv("dbfs:/databricks-datasets/asa/airlines/2*.csv", header=True, inferSchema=True)\
.write.mode("append").saveAsTable("flight_data_z")

---
## 3. Z-Ordering por Year e Origin

O `OPTIMIZE ... ZORDER BY` executa dois passos em sequência:  
1. **Compacta** os arquivos pequenos (bin-packing, igual ao `OPTIMIZE` simples)  
2. **Reordena** as linhas dentro dos arquivos resultantes usando o algoritmo Z

In [ ]:
%%sql
-- ⏱ Operação demorada: precisa ler todos os dados, aplicar o algoritmo Z
-- e reescrever os arquivos na nova ordem
OPTIMIZE flight_data_z
ZORDER BY (Year, Origin);

### Verificando o resultado

In [ ]:
%%sql
-- ⚠️ Não há coluna que indique que a tabela passou por Z-Ordering
-- DESCRIBE DETAIL não expõe essa informação diretamente
DESC DETAIL flight_data_z;

In [ ]:
%%sql
-- O histórico mostra a operação OPTIMIZE com Z-Ordering
-- mas consultar o histórico para isso não é prático para um usuário regular
DESC HISTORY flight_data_z;

### Limpando os arquivos antigos para ver a estrutura final

In [ ]:
%%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM flight_data_z RETAIN 0 HOURS;

In [ ]:
-- O Z-Ordering compactou todos os arquivos em apenas 3 Parquets
-- Cada arquivo com mais de 300 MB — dados co-localizados por Year e Origin
%fs
ls dbfs:/user/hive/warehouse/flight_data_z

In [ ]:
%%sql
-- O commit do Z-Ordering mostra:
-- - Maioria das entradas: 'remove' (arquivos anteriores descartados)
-- - Poucas entradas: 'add' (3 novos arquivos Z-ordenados adicionados)
SELECT * FROM JSON.`dbfs:/user/hive/warehouse/flight_data_z/_delta_log/00000000000000000002.json`

---
## 4. Alterando as colunas do Z-Ordering

O Z-Ordering pode ser refeito a qualquer momento com colunas diferentes.  
Cada re-execução reescreve os arquivos com a nova ordenação — sem precisar recriar a tabela.

> **Atenção:** como os dados mudam ao longo do tempo (novos inserts, updates),  
> o Z-Ordering precisa ser re-executado periodicamente para manter a eficiência.

In [ ]:
%%sql
-- Reordena pelos aeroportos de origem e destino
-- Ideal para queries que filtram por rotas específicas: WHERE Origin = 'ATL' AND Dest = 'LAX'
OPTIMIZE flight_data_z
ZORDER BY (Origin, Dest);

In [ ]:
%%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM flight_data_z RETAIN 0 HOURS;

In [ ]:
-- Ainda 3 arquivos — mas com tamanhos diferentes do Z-Order anterior
-- Os dados foram redistribuídos de acordo com a nova ordem: Origin + Dest
%fs
ls dbfs:/user/hive/warehouse/flight_data_z

> **Por que os tamanhos mudaram?**  
> O Z-Ordering por `(Year, Origin)` co-localiza dados que são similares em ano e aeroporto de origem.  
> O Z-Ordering por `(Origin, Dest)` co-localiza dados que são similares em rota (origem → destino).  
> A compressão Parquet é mais eficiente quando dados similares ficam juntos — por isso os tamanhos diferem.

---

## 5. Partitioning + Z-Ordering (caso mais comum em produção)

A combinação mais poderosa é usar **particionamento para baixa cardinalidade**  
e **Z-Ordering dentro de cada partição para alta cardinalidade**.

Isso resolve dois problemas ao mesmo tempo:
- O particionamento por `Year` elimina partições inteiras nas queries com filtro de ano
- O Z-Ordering por `Origin` dentro de cada partição reduz os falsos positivos nas queries de rota

```
  flight_data_year_z/
  ├── _delta_log/
  ├── Year=2007/
  │     └── part-001.parquet  ← Z-Ordenado por Origin
  │     └── part-002.parquet  ← Z-Ordenado por Origin
  └── Year=2008/
        └── part-003.parquet  ← Z-Ordenado por Origin
        └── part-004.parquet  ← Z-Ordenado por Origin

  WHERE Year = 2008 AND Origin = 'ATL'
      │
      ├── Partição elimina Year=2007/ completamente
      └── Z-Order elimina arquivos de Year=2008/ sem dados de ATL
```

In [ ]:
%%sql
DROP TABLE IF EXISTS flight_data_year_z;

CREATE TABLE flight_data_year_z (
    Year INT, Month INT, DayOfMonth INT, DayOfWeek INT, DepTime STRING, CRSDepTime INT, ArrTime STRING, CRSArrTime INT, UniqueCarrier STRING, FlightNum INT, TailNum STRING, ActualElapsedTime STRING, CRSElapsedTime STRING, AirTime STRING, ArrDelay STRING, DepDelay STRING, Origin STRING, Dest STRING, Distance INT, TaxiIn STRING, TaxiOut STRING, Cancelled INT, CancellationCode STRING, Diverted INT, CarrierDelay STRING, WeatherDelay STRING, NASDelay STRING, SecurityDelay STRING, LateAircraftDelay STRING
)
PARTITIONED BY (Year);

In [ ]:
%python
spark.read.csv("dbfs:/databricks-datasets/asa/airlines/2*.csv", header=True, inferSchema=True)\
.write.mode("append").saveAsTable("flight_data_year_z")

In [ ]:
-- Estado antes do Z-Ordering: múltiplos arquivos pequenos dentro da partição Year=2008
%fs
ls dbfs:/user/hive/warehouse/flight_data_year_z/Year=2008/

### Aplicando Z-Ordering em uma partição específica

O `OPTIMIZE ... WHERE` permite otimizar **apenas uma partição** — útil para otimização incremental  
em pipelines que ingerem dados por período (ex: otimizar apenas o mês/ano recém-chegado).

In [ ]:
%%sql
-- WHERE Year = 2008 → aplica OPTIMIZE + Z-Order APENAS na partição Year=2008
-- Year=2007 permanece inalterada
OPTIMIZE flight_data_year_z 
WHERE Year = 2008
ZORDER BY (Origin);

In [ ]:
%%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM flight_data_year_z RETAIN 0 HOURS;

In [ ]:
-- Após o OPTIMIZE: Year=2008 compactada e Z-Ordenada por Origin
-- Poucos arquivos grandes, co-localizados por aeroporto de origem
%fs
ls dbfs:/user/hive/warehouse/flight_data_year_z/Year=2008/

In [ ]:
-- Year=2007 não foi tocada — ainda com os arquivos originais sem Z-Ordering
%fs
ls dbfs:/user/hive/warehouse/flight_data_year_z/Year=2007/

> **Comparação direta:**  
> `Year=2008/` → poucos arquivos grandes, Z-Ordenados por Origin → queries rápidas  
> `Year=2007/` → muitos arquivos pequenos, sem Z-Order → pequeno files problem ainda presente  
>
> Isso mostra o valor da otimização **incremental por partição**: em vez de reprocessar  
> toda a tabela a cada ciclo, otimiza-se apenas a partição com dados novos.

---

## 6. Z-Ordering — Key Takeaways

```
┌──────────────────────────────────────────────────────────────────┐
│                    Z-ORDERING — PONTOS CHAVE                     │
│                                                                  │
│  ✅ Geralmente mais eficiente que particionamento puro           │
│     → Evita over-partitioning e small files por partição         │
│     → Skip de dados mais preciso (menos falsos positivos)        │
│                                                                  │
│  ✅ Mais colunas no ZORDER BY = mais benefício                   │
│     → Cada coluna adicional melhora a co-localização             │
│                                                                  │
│  ✅ Ideal para colunas de alta cardinalidade                     │
│     → Especialmente colunas usadas em JOINs e filtros frequentes │
│                                                                  │
│  ✅ Combinação poderosa: Partitioning + Z-Ordering               │
│     → Particiona por baixa cardinalidade (Year)                  │
│     → Z-Ordena por alta cardinalidade dentro da partição (Origin)│
│                                                                  │
│  ⚠️  As colunas do ZORDER BY devem estar entre as primeiras 32   │
│     (limite do _delta_log para coleta de estatísticas)           │
│                                                                  │
│  ⚠️  Não é automático — deve ser re-executado periodicamente     │
│     À medida que novos dados chegam, a ordenação se degrada      │
│                                                                  │
│  ⚠️  Pode ser aplicado dentro de partições específicas           │
│     OPTIMIZE table WHERE partition_col = value ZORDER BY (...)   │
└──────────────────────────────────────────────────────────────────┘
```

### Comparação final — todas as estratégias de otimização

| Estratégia | Cardinalidade | Automático | Reescrita da tabela | Subdiretórios | Melhor para |
|---|---|---|---|---|---|
| **File Skipping** | Qualquer | ✅ Sempre | ❌ | ❌ | Qualquer tabela, sem custo |
| **OPTIMIZE** | N/A | ⚡ Auto Compact | ❌ | ❌ | Small files problem |
| **Partitioning** | Baixa | ❌ | ✅ | ✅ | Tabelas > 1TB, filtros frequentes por partição |
| **Z-Ordering** | Alta | ❌ | ❌ | ❌ | Colunas de JOINs, filtros em alta cardinalidade |
| **Partition + Z-Order** | Baixa + Alta | ❌ | ✅ (partição) | ✅ | Produção: melhor dos dois mundos |

### Comandos de referência

| Objetivo | Comando |
|---|---|
| Z-Ordering em tabela inteira | `OPTIMIZE table ZORDER BY (col1, col2)` |
| Z-Ordering em partição específica | `OPTIMIZE table WHERE year = 2008 ZORDER BY (col1)` |
| Ver histórico de otimizações | `DESC HISTORY table` |
| Limpar arquivos antigos após OPTIMIZE | `VACUUM table RETAIN N HOURS` |